# `final_correlation` — behaviour × neural, across two mice

Loads the aligned bundles, **proves the interpolation and alignment are exactly right**,
audits what may legitimately be compared across animals, then correlates behaviour with
ROI activity.

Design: [16_final_correlation.md](16_final_correlation.md).

## Three tiers of input
| tier | source | why |
|---|---|---|
| 1 | `bundles/<run>_aligned.npz` | the aligned matrices |
| 2 | `meta["ca_folder"]` | `F0`, `mean_img`, `corr_map`, `roi_map.tif`, `params.json` — **not carried in the bundle** |
| 3 | `meta["beh_npz"]` | camera-rate `motion_energy` / `traces`, before interpolation |

## The gate
Stage 1b re-derives the interpolation from tier 3 and checks it sample by sample.
Because both runs have **integer rate ratios**, coincident samples must be *bit-identical*
copies of the source — `np.array_equal`, not `np.allclose`. Nothing downstream runs on a
bundle that fails.

⚠ Every correlation below is between two time series. If the mapping between them is
wrong, every number is wrong **and nothing in the output looks unusual.** That is what
stage 1b exists to rule out.

In [ ]:
# ========================= THE ONLY CELL YOU EDIT =========================
from pathlib import Path

BUNDLE_DIR = Path("/grid/courses/data/imagcourse/GECI_Project_Analyzed/bundles")
FIG_DIR    = BUNDLE_DIR / "analysis"        # figures + results land here

RUN_NAMES  = ["thormouse1_spont", "thormouse2_spont"]

PRIMARY_BOX = "whisker_pad"   # the arousal regressor everything keys on
ENV_S       = 0.33            # conditioned-envelope window, s -- must match the bundles
MAX_LAG_S   = 10.0            # lag range to report
NULL_MIN_S  = 15.0            # |lag| beyond this is treated as the null distribution
REPO_DIR    = None            # folder holding whisk_bouts.py. None = autodetect.

In [ ]:
import json, sys, traceback
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

for _n in ("BUNDLE_DIR", "RUN_NAMES", "PRIMARY_BOX", "ENV_S", "REPO_DIR"):
    if _n not in globals():
        raise NameError(f"{_n} undefined -- run the config cell above first.")
FIG_DIR = Path(FIG_DIR); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("bundles ->", BUNDLE_DIR, "\nfigures ->", FIG_DIR)

# condition() is imported, never reimplemented (10_behavior_motifs.md §7).
_cand = ([Path(REPO_DIR)] if REPO_DIR else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / "behavioral motif analysis",
    Path.cwd().parent / "behavioral motif analysis"]
condition = None
for _d in _cand:
    if (_d / "whisk_bouts.py").exists():
        sys.path.insert(0, str(_d))
        from whisk_bouts import condition                       # noqa: E402
        print(f"condition() <- {_d/'whisk_bouts.py'}")
        break
if condition is None:
    print("*** whisk_bouts.py NOT FOUND. If the bundles also lack 'beh_env', the\n"
          "    analysis falls back to RAW motion energy, which carries flicker to the\n"
          "    camera Nyquist. Set REPO_DIR. ***")

In [ ]:
CA_FILES = ("F0.npy", "mean_img.npy", "corr_map.npy", "shifts_yx.npy",
            "traces_raw.npy", "roi_npix.npy")


def load_run(name):
    """Tier 1 + 2 + 3 for one run. Missing tier-2/3 files are recorded, never fatal."""
    p = BUNDLE_DIR / f"{name}_aligned.npz"
    if not p.exists():
        raise FileNotFoundError(f"bundle not found: {p}")
    z = np.load(p, allow_pickle=True)
    R = {"name": name, "bundle": str(p), "meta": json.loads(str(z["meta"]))}
    for k in z.files:
        if k != "meta":
            R[k] = z[k]
    R["beh_names"] = [str(s) for s in z["beh_names"]]
    R["M_names"] = [str(s) for s in z["M_names"]]
    R["arrays"] = [k for k in z.files if k != "meta"]

    ca = Path(R["meta"]["ca_folder"]); R["ca"], R["ca_missing"] = {}, []
    for fn in CA_FILES:
        (R["ca"].__setitem__(fn[:-4], np.load(ca / fn)) if (ca / fn).exists()
         else R["ca_missing"].append(fn))
    if (ca / "roi_map.tif").exists():
        try:
            import tifffile
            R["ca"]["roi_map"] = tifffile.imread(ca / "roi_map.tif")
        except Exception as e:
            R["ca_missing"].append(f"roi_map.tif ({e})")
    else:
        R["ca_missing"].append("roi_map.tif")
    R["ca"]["params"] = (json.loads((ca / "params.json").read_text())
                         if (ca / "params.json").exists() else {})
    if not R["ca"]["params"]:
        R["ca_missing"].append("params.json")

    s = Path(R["meta"]["beh_npz"])
    if not s.exists():
        raise FileNotFoundError(f"{name}: source box traces gone: {s}\n"
                                f"Tier 3 is required -- stage 1b cannot verify without it.")
    d = np.load(s, allow_pickle=True)
    R["src"] = dict(path=str(s), names=[str(x) for x in d["box_names"]],
                    me=np.asarray(d["motion_energy"], float),
                    lum=np.asarray(d["traces"], float))
    R["src"]["n_cam"] = R["src"]["me"].shape[1]
    return R


RUNS_D, load_fail = {}, {}
for nm in RUN_NAMES:
    try:
        RUNS_D[nm] = load_run(nm)
        print(f"loaded {nm}")
    except Exception as e:
        load_fail[nm] = f"{type(e).__name__}: {e}"
        print(f"FAILED {nm}: {load_fail[nm]}")

In [ ]:
# ============================ STAGE 1 -- inventory ============================
def fmt(a):
    a = np.asarray(a)
    if a.dtype.kind in "fiu" and a.size:
        return (f"{str(a.shape):>16s} {str(a.dtype):>8s}  "
                f"min {np.nanmin(a):11.4g}  med {np.nanmedian(a):11.4g}  "
                f"max {np.nanmax(a):11.4g}  "
                f"nan {int(np.isnan(a).sum()) if a.dtype.kind=='f' else 0}")
    return f"{str(a.shape):>16s} {str(a.dtype):>8s}"


for nm, R in RUNS_D.items():
    m, T = R["meta"], len(R["t"])
    print("=" * 96)
    print(f"{nm}   T={T}  n_roi={m['n_roi']}  n_box={m['n_box']}  "
          f"{m['tseries_duration_s']:.1f}s of t-series")
    print(f"  map      {m['map_mode']}  2P {1/m['tseries_frame_period_s']:.4f} Hz  "
          f"cam {m['cam_fps_used']:.4f} Hz  interp {m['interp_factor']:.2f}x")
    print(f"  coverage {m['coverage_end_s']:.2f}s   dropped {m['n_2p_dropped']} of "
          f"{m['n_2p_full']}   anchors {m['on']}..{m['off']}   "
          f"origin unc +-{m['origin_uncertainty_s']*1000:.0f} ms")
    print(f"  boxes    {R['beh_names']}")
    print(f"  conditioned env in bundle: {'YES' if 'beh_env' in R else 'NO'}")

    print("  -- tier 1 arrays " + "-" * 60)
    for k in sorted(R["arrays"]):
        if k in ("beh_names", "M_names"):
            print(f"    {k:12s} {str(np.asarray(R[k]).shape):>16s}  {list(R[k])[:6]}")
        else:
            print(f"    {k:12s} {fmt(R[k])}")

    print("  -- tier 2 (v6 outputs NOT in the bundle) " + "-" * 34)
    for k, v in R["ca"].items():
        print(f"    {k:12s} {fmt(v) if k != 'params' else str(v)[:110]}")
    if R["ca_missing"]:
        print(f"    *** MISSING: {R['ca_missing']} ***")

    print("  -- tier 3 (camera-rate source) " + "-" * 44)
    print(f"    n_cam {R['src']['n_cam']}   boxes {R['src']['names']}")
    print(f"    me    {fmt(R['src']['me'])}")

    # --- hard structural checks -------------------------------------------
    bad = []
    tvec = np.asarray(R["t"])
    for k in ("dff", "beh_me", "beh_lum", "M") + (("beh_env",) if "beh_env" in R else ()):
        if np.asarray(R[k]).shape[-1] != T:
            bad.append(f"{k} last axis {np.asarray(R[k]).shape[-1]} != T {T}")
    if tvec[0] != 0:                       bad.append(f"t[0]={tvec[0]} != 0")
    if not np.all(np.diff(tvec) > 0):      bad.append("t not strictly increasing")
    if R["src"]["names"] != R["beh_names"]:
        bad.append(f"tier3 boxes {R['src']['names']} != bundle {R['beh_names']}")
    blocks = [("roi", R["dff"]), ("me", R["beh_me"]), ("lum", R["beh_lum"])] + \
             ([("env", R["beh_env"])] if "beh_env" in R else [])
    if not np.array_equal(np.vstack([b[1] for b in blocks]), R["M"]):
        bad.append("M != vstack(dff, beh_me, beh_lum[, beh_env])")
    mu, sd = R["dff"].mean(1), R["dff"].std(1)
    if np.abs(mu).max() < 1e-6 and np.abs(sd - 1).max() < 1e-3:
        bad.append("dff looks Z-SCORED -- amplitudes are not comparable")
    print(f"  -- structural checks: "
          + ("ALL PASS" if not bad else f"{len(bad)} FAILURE(S)"))
    for b in bad:
        print(f"       *** {b}")
    R["stage1_ok"] = not bad
    print(f"  artifact frames: {int(np.asarray(R['artifact']).sum())} of {T}")

In [ ]:
# ================ STAGE 1b -- THE GATE: interpolation & alignment ================
# Re-derives the mapping from tier 3 and checks it sample by sample. Coincident
# samples must be BIT-IDENTICAL copies of the source (integer rate ratios), so these
# are equality assertions, not tolerance assertions.
def repaired_me(src_me, on):
    me = src_me.copy()
    if me.shape[1] > 1:
        me[:, 0] = me[:, 1]                 # 0 by construction (no previous frame)
    me[:, on] = me[:, on + 1]               # frame `on` holds the dark->bright step
    return me


def verify(R):
    m, t = R["meta"], np.asarray(R["t"])
    on, off, cam = m["on"], m["off"], m["cam_fps_used"]
    n_cam, T = R["src"]["n_cam"], len(t)
    me = repaired_me(R["src"]["me"], on)
    lum = R["src"]["lum"]
    t_cam = (np.arange(n_cam) - on) * (1.0 / cam)
    res = []

    # -- 1. independent reconstruction ------------------------------------
    rec_me = np.vstack([np.interp(t, t_cam, r) for r in me])
    rec_lum = np.vstack([np.interp(t, t_cam, r) for r in lum])
    d1 = max(np.abs(rec_me - R["beh_me"]).max(), np.abs(rec_lum - R["beh_lum"]).max())
    res.append(("1 reconstruction from tier 3", d1 == 0.0, f"max|d| = {d1:.3e}"))

    # -- 2. coincident samples are exact copies ---------------------------
    q = m["tseries_frame_period_s"] and (1.0 / m["tseries_frame_period_s"]) / cam
    qi = int(round(q))
    if abs(q - qi) > 1e-9 or qi < 1:
        res.append(("2 coincident samples", None, f"rate ratio {q:.4f} not integer -- "
                                                  f"test not applicable"))
    else:
        k = np.arange(0, T, qi)
        f = on + k // qi
        ok = f[-1] <= n_cam - 1
        d2 = np.abs(R["beh_me"][:, k] - me[:, f]).max() if ok else np.inf
        res.append((f"2 coincident samples ({len(k)} of {T}, ratio {qi}:1)",
                    ok and np.array_equal(R["beh_me"][:, k], me[:, f]),
                    f"max|d| = {d2:.3e}  (must be exactly 0)"))

    # -- 3. interpolated samples are the exact linear blend ---------------
    if abs(q - qi) < 1e-9 and qi > 1:
        k = np.arange(T)
        ff = k / qi
        f0 = np.floor(ff).astype(int)
        r = ff - f0
        f0 = on + f0
        inside = f0 + 1 <= n_cam - 1
        exp = ((1 - r[inside]) * me[:, f0[inside]] + r[inside] * me[:, f0[inside] + 1])
        d3 = np.abs(R["beh_me"][:, k[inside]] - exp).max()
        res.append(("3 interpolated = linear blend (index arithmetic)", d3 < 1e-9,
                    f"max|d| = {d3:.3e}  (np.interp uses the slope form -- ~1e-11 "
                    f"expected, not 0)"))

    # -- 4. no overshoot, no extrapolation --------------------------------
    hi = min(off, n_cam - 1)
    lo_s, hi_s = me[:, on:hi + 1].min(1), me[:, on:hi + 1].max(1)
    ov = max((lo_s - R["beh_me"].min(1)).max(), (R["beh_me"].max(1) - hi_s).max())
    cov = (off - on) / cam
    res.append(("4 no overshoot / no extrapolation",
                ov <= 1e-9 and t[-1] <= cov + 1e-9,
                f"max overshoot {ov:.3e}   t[-1] {t[-1]:.3f} <= coverage {cov:.3f}"))

    # -- 5. anchors sit on real laser edges -------------------------------
    if "laser_trigger" in R["src"]["names"]:
        L = lum[R["src"]["names"].index("laser_trigger")]
        Lm = L.copy()
        Lm[1:-1] = np.median(np.stack([L[:-2], L[1:-1], L[2:]]), 0)
        dL = np.diff(Lm)
        up = int(np.argmax(dL)) + 1
        inside = L[on:hi + 1]
        outside = np.r_[L[:on], L[hi + 1:]]
        sep = ((np.median(inside) - np.median(outside)) /
               max(np.median(np.abs(L - np.median(L))) * 1.4826, 1e-9))
        stopped = m["map_mode"] == "single_anchor_known_fps" and off >= n_cam - 5
        dn = int(np.argmin(dL[up + 1:])) + up + 1
        res.append(("5 anchors on real laser edges", abs(up - on) <= 2 and sep > 5,
                    f"detected ON {up} vs configured {on}; inside-vs-outside "
                    f"{sep:.1f} robust SD; "
                    + (f"camera stopped early, no OFF edge expected (off={off}, "
                       f"n_cam={n_cam})" if stopped else f"detected OFF {dn} vs {off}")))
    else:
        res.append(("5 anchors on real laser edges", None, "no laser_trigger box"))

    # -- 6. coverage arithmetic from meta ---------------------------------
    t_full = np.arange(m["n_2p_full"]) * m["tseries_frame_period_s"]
    keep = int(np.searchsorted(t_full, cov, side="right"))
    res.append(("6 coverage arithmetic self-consistent",
                abs(m["coverage_end_s"] - cov) < 1e-9 and keep == T
                and m["n_2p_dropped"] == m["n_2p_full"] - T,
                f"T {T} == {keep}, dropped {m['n_2p_dropped']} == "
                f"{m['n_2p_full'] - T}"))
    return res


GATE = {}
for nm, R in RUNS_D.items():
    print("=" * 96)
    print(f"STAGE 1b — {nm}")
    try:
        res = verify(R)
    except Exception as e:
        print(f"  *** verification itself crashed: {type(e).__name__}: {e}")
        traceback.print_exc(); GATE[nm] = False; continue
    for label, ok, detail in res:
        tag = "PASS" if ok else ("n/a " if ok is None else "FAIL")
        print(f"  [{tag}] {label:52s} {detail}")
    hard = [ok for _, ok, _ in res if ok is not None]
    GATE[nm] = all(hard)
    R["gate_ok"] = GATE[nm]
    print(f"  --> {'GATE PASSED' if GATE[nm] else '*** GATE FAILED — no figure may be '
          'built from this bundle ***'}")

OK_RUNS = [n for n in RUNS_D if GATE.get(n) and RUNS_D[n].get("stage1_ok")]
print("\n" + "=" * 96)
print(f"runs cleared for analysis: {OK_RUNS}")
if len(OK_RUNS) < len(RUN_NAMES):
    print(f"*** BLOCKED: {sorted(set(RUN_NAMES) - set(OK_RUNS))} ***")

In [ ]:
# ==================== STAGE 2 -- cross-animal audit ====================
def acf(x, nlag):
    x = np.asarray(x, float); x = x - x.mean()
    n = len(x); nfft = 1 << int(np.ceil(np.log2(2 * n)))
    f = np.fft.rfft(x, nfft)
    a = np.fft.irfft(f * np.conj(f), nfft)[:nlag + 1]
    return a / a[0]


def n_eff(x, y, fs, max_lag_s=30.0):
    """Bartlett: Var(r) ~ (1/T) sum_k rho_x(k) rho_y(k). Truncate the sum at the first
    zero crossing of either autocorrelation -- summing past it adds only noise."""
    L = min(int(max_lag_s * fs), len(x) // 4)
    rx, ry = acf(x, L), acf(y, L)
    k = 1
    while k <= L and rx[k] > 0 and ry[k] > 0:
        k += 1
    s = float(np.sum(rx[1:k] * ry[1:k]))
    return len(x) / max(1.0 + 2.0 * s, 1.0), k / fs


box_sets = {n: set(RUNS_D[n]["beh_names"]) for n in OK_RUNS}
COMMON = sorted(set.intersection(*box_sets.values())) if box_sets else []
print("boxes per run:")
for n in OK_RUNS:
    print(f"  {n:20s} {sorted(box_sets[n])}")
print(f"\ncommon to all: {COMMON}")
for n in OK_RUNS:
    drop = sorted(box_sets[n] - set(COMMON))
    if drop:
        print(f"  {n}: dropped from cross-animal work -> {drop}")
if PRIMARY_BOX not in COMMON:
    print(f"\n*** '{PRIMARY_BOX}' is NOT common to all runs -- the primary analysis is "
          f"single-animal. ***")

print("\ncomparability:")
print(f"  {'run':20s} {'T':>7s} {'dur s':>8s} {'cam Hz':>7s} {'interp':>7s} "
      f"{'n_roi':>6s} {'beh DOF':>8s}")
for n in OK_RUNS:
    m = RUNS_D[n]["meta"]
    print(f"  {n:20s} {m['n_2p']:7d} {m['coverage_end_s']:8.1f} "
          f"{m['cam_fps_used']:7.2f} {m['interp_factor']:6.2f}x {m['n_roi']:6d} "
          f"{m['n_2p']/m['interp_factor']:8.0f}")
print("  'beh DOF' = independent behavioural samples. Differs between animals whenever\n"
      "  interp factors differ -- never compare sample-counting statistics directly.")

In [ ]:
# ============ STAGE 3 -- regressors: conditioned envelope on the 2P grid ============
# Prefer the bundle's beh_env. If absent, rebuild it here from tier 3 at the CAMERA
# rate and interpolate -- conditioning a already-upsampled trace would smooth over
# samples that carry no independent information.
for n in OK_RUNS:
    R = RUNS_D[n]; m = R["meta"]
    if "beh_env" in R:
        R["env"] = np.asarray(R["beh_env"]); R["env_src"] = "bundle"
    elif condition is not None:
        on, cam = m["on"], m["cam_fps_used"]
        me = repaired_me(R["src"]["me"], on)
        t_cam = (np.arange(R["src"]["n_cam"]) - on) * (1.0 / cam)
        R["env"] = np.vstack([np.interp(R["t"], t_cam, condition(r, cam, ENV_S))
                              for r in me])
        R["env_src"] = f"recomputed here (condition, env_s={ENV_S})"
    else:
        R["env"] = np.asarray(R["beh_me"]); R["env_src"] = "*** RAW ME fallback ***"
    print(f"{n:20s} env <- {R['env_src']}   shape {R['env'].shape}")

In [ ]:
# ============ STAGE 4 -- cross-correlation and lag curves ============
def xcorr(D, b, fs, max_lag_s, null_min_s):
    """D: n_roi x T (z-scored). b: T (z-scored). Returns lags (s), cc [n_roi x n_lag],
    r0 [n_roi], and the null distribution taken from |lag| > null_min_s.

    Sign: positive lag = neural LAGS behaviour, i.e. behaviour leads."""
    D = (D - D.mean(1, keepdims=True)) / np.maximum(D.std(1, keepdims=True), 1e-12)
    b = (b - b.mean()) / max(b.std(), 1e-12)
    T = D.shape[1]
    nfft = 1 << int(np.ceil(np.log2(2 * T)))
    F = np.fft.rfft(D, nfft, axis=1)
    G = np.fft.rfft(b, nfft)
    cc_full = np.fft.irfft(F * np.conj(G), nfft, axis=1) / T
    L = int(max_lag_s * fs)
    cc = np.concatenate([cc_full[:, -L:], cc_full[:, :L + 1]], axis=1)
    lags = np.arange(-L, L + 1) / fs
    Ln = int(null_min_s * fs)
    null = np.concatenate([cc_full[:, Ln:T - Ln]], axis=1)
    return lags, cc, cc_full[:, 0].copy(), null


RES = {}
for n in OK_RUNS:
    R = RUNS_D[n]; m = R["meta"]; fs = 1.0 / m["tseries_frame_period_s"]
    keep = ~np.asarray(R["artifact"], bool)          # drop laser-transition frames
    dff = np.asarray(R["dff"])[:, keep]
    RES[n] = {}
    for box in COMMON:
        if box == "laser_trigger":
            continue
        b = R["env"][R["beh_names"].index(box)][keep]
        lags, cc, r0, null = xcorr(dff, b, fs, MAX_LAG_S, NULL_MIN_S)
        ne, tau = n_eff(dff.mean(0), b, fs)
        floor = 2.0 / np.sqrt(max(ne - 3, 1))
        RES[n][box] = dict(lags=lags, cc=cc, r0=r0, null=null, n_eff=ne, floor=floor)
        pk = int(np.abs(cc).mean(0).argmax())
        print(f"{n:20s} {box:14s} |r| median {np.median(np.abs(r0)):.3f}  "
              f"max {np.abs(r0).max():.3f}  above floor "
              f"{int((np.abs(r0) > floor).sum())}/{len(r0)}  "
              f"floor +-{floor:.3f}  peak lag {lags[pk]:+.2f}s")
print("\n'floor' is a rough 2-sigma guide from autocorrelation-corrected sample count --\n"
      "a line on the figure so nothing below it gets called an effect. Not a p-value.")

In [ ]:
# ============================== FIGURES ==============================
box = PRIMARY_BOX if PRIMARY_BOX in COMMON else (COMMON[1] if len(COMMON) > 1 else None)
if OK_RUNS and box and all(box in RES[n] for n in OK_RUNS):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

    # --- 1. per-ROI r distribution, per mouse -------------------------------
    for n in OK_RUNS:
        d = RES[n][box]
        ax[0].hist(d["r0"], bins=np.linspace(-.6, .6, 33), alpha=.55,
                   label=f"{n} (n={len(d['r0'])})")
    fl = max(RES[n][box]["floor"] for n in OK_RUNS)
    for s in (-fl, fl):
        ax[0].axvline(s, color="k", ls="--", lw=1)
    ax[0].axvline(0, color="0.6", lw=.8)
    ax[0].set(xlabel=f"r (ROI × {box} envelope, lag 0)", ylabel="ROIs",
              title=f"per-ROI correlation\ndashed = 2σ floor (±{fl:.2f})")
    ax[0].legend(fontsize=7)

    # --- 2. population lag curve -------------------------------------------
    for n in OK_RUNS:
        d = RES[n][box]
        R = RUNS_D[n]
        keep = ~np.asarray(R["artifact"], bool)
        mu = np.asarray(R["dff"])[:, keep].mean(0, keepdims=True)
        b = R["env"][R["beh_names"].index(box)][keep]
        lg, cc, _, nl = xcorr(mu, b, 1 / R["meta"]["tseries_frame_period_s"],
                              MAX_LAG_S, NULL_MIN_S)
        ax[1].plot(lg, cc[0], lw=1.2, label=n)
        ax[1].fill_between(lg, -np.percentile(np.abs(nl), 95),
                           np.percentile(np.abs(nl), 95), color="0.8", alpha=.35, lw=0)
    ax[1].axvline(0, color="0.6", lw=.8); ax[1].axhline(0, color="0.6", lw=.8)
    ax[1].set(xlabel="lag (s)   >0: behaviour leads", ylabel="r",
              title=f"FOV-mean ΔF/F × {box}\nshaded = 95% of the |lag|>"
                    f"{NULL_MIN_S:.0f}s null")
    ax[1].legend(fontsize=7)

    # --- 3. fraction above the floor, per box, per mouse -------------------
    boxes = [b for b in COMMON if b != "laser_trigger"]
    w = .38
    for i, n in enumerate(OK_RUNS):
        frac = [(np.abs(RES[n][b]["r0"]) > RES[n][b]["floor"]).mean() for b in boxes]
        ax[2].bar(np.arange(len(boxes)) + i * w, frac, w, label=n)
    ax[2].set(xticks=np.arange(len(boxes)) + w / 2, ylabel="fraction of ROIs",
              title="ROIs above the 2σ floor\n(the one honest cross-animal scalar)")
    ax[2].set_xticklabels(boxes, rotation=20, ha="right", fontsize=8)
    ax[2].legend(fontsize=7)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "correlation_summary.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIG_DIR / "correlation_summary.pdf", bbox_inches="tight")
    print("wrote", FIG_DIR / "correlation_summary.png")
    plt.show()
else:
    print(f"'{PRIMARY_BOX}' not available in every cleared run -- summary figure skipped.")

In [ ]:
# ======== FOV map: where the modulated ROIs are (needs tier 2 mean_img + roi_xy) ======
for n in OK_RUNS:
    R = RUNS_D[n]
    if box not in RES[n] or "roi_xy" not in R or "mean_img" not in R["ca"]:
        print(f"{n}: skipped (roi_xy or mean_img missing)"); continue
    xy, r0 = np.asarray(R["roi_xy"]), RES[n][box]["r0"]
    if len(xy) != len(r0):
        print(f"{n}: roi_xy has {len(xy)} rows but {len(r0)} ROIs -- skipped"); continue
    img = R["ca"]["mean_img"]
    fig, a = plt.subplots(figsize=(6, 5.6))
    a.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    v = np.abs(r0).max() or 1
    s = a.scatter(xy[:, 0], xy[:, 1], c=r0, cmap="coolwarm", vmin=-v, vmax=v,
                  s=44, edgecolor="k", linewidth=.4)
    plt.colorbar(s, ax=a, label=f"r with {box}")
    a.set(title=f"{n} — ROI modulation on the FOV", xticks=[], yticks=[])
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{n}_fov_map.png", dpi=200, bbox_inches="tight")
    print("wrote", FIG_DIR / f"{n}_fov_map.png")
    plt.show()

In [ ]:
# ==== TEST 7 -- empirical alignment check. Corroboration, NOT proof (see 16 §2b) ====
SHIFTS = [-50, -20, -10, -5, -2, -1, 0, 1, 2, 5, 10, 20, 50]     # camera frames
if not OK_RUNS:
    raise SystemExit("no run cleared stage 1b -- nothing to plot")
fig, a = plt.subplots(figsize=(7, 4.2))
for n in OK_RUNS:
    R = RUNS_D[n]; m = R["meta"]
    on, cam, fs = m["on"], m["cam_fps_used"], 1 / m["tseries_frame_period_s"]
    me = repaired_me(R["src"]["me"], on)
    if PRIMARY_BOX not in R["beh_names"]:
        continue
    row = me[R["beh_names"].index(PRIMARY_BOX)]
    env_c = condition(row, cam, ENV_S) if condition is not None else row
    mu = np.asarray(R["dff"]).mean(0)
    mu = (mu - mu.mean()) / max(mu.std(), 1e-12)
    vals = []
    for s in SHIFTS:
        t_cam = (np.arange(len(row)) - (on + s)) * (1.0 / cam)
        e = np.interp(R["t"], t_cam, env_c)
        e = (e - e.mean()) / max(e.std(), 1e-12)
        vals.append(abs(float((e * mu).mean())))
    a.plot(np.array(SHIFTS) / cam, vals, "o-", ms=4, lw=1.2, label=n)
a.axvline(0, color="0.6", lw=.8)
a.set(xlabel="anchor shift (s)", ylabel=f"|r|  (FOV-mean ΔF/F × {PRIMARY_BOX})",
      title="Test 7 — alignment shift curve\nflat within ±0.5 s even when correct; "
            "a flat curve is ambiguous")
a.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "alignment_shift_curve.png", dpi=200, bbox_inches="tight")
print("wrote", FIG_DIR / "alignment_shift_curve.png")
plt.show()

## What may and may not be said

**May.** Per-ROI correlations with the whisking envelope, within an animal, against a
2σ floor computed from Bartlett-corrected effective N. Lag structure at ≥0.5 s
resolution. The fraction of ROIs exceeding the floor, compared between the two mice.

**May not.**

- **Not locomotion, not speed.** `paw` and `wheel` are movement energy. Zero sustained
  paw+wheel episodes were found, 85% of paw-active frames are grooming, and mouse 2 has
  no `paw_at_nose` to veto grooming with.
- **Not fine timing or causality.** ±100 ms common-mode origin uncertainty, and GCaMP7s
  kinetics blur it further. Lag claims at ≥0.5 s only.
- **Not an odour effect.** No PB/TMT contrast exists in these bundles.
- **Not "n = 2 therefore general."** Two animals, one FOV each. The per-ROI
  distributions are *within*-animal; the animal is the replicate, and there are two.
- **Not a pupil result for mouse 2.** No `eye` box in that recording.
- **Not a p-value from `scipy.stats.pearsonr`.** It assumes ~27 000 independent samples;
  the true figure is a few hundred. Use the shift null computed in stage 4.

**Effective N differs between the animals** — mouse 1's behaviour is native 30 Hz,
mouse 2's is 15 Hz upsampled. Same frame count, half the independent information. Stage 2
prints both; never compare a sample-counting statistic across the two without it.